In [1]:
import gc
import warnings
from typing import Dict, Any, List

# --- Third‑party libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import scvi
import torch
import tensorflow as tf
#import anndata
from anndata import AnnData
from matplotlib.lines import Line2D
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from scvi.dataloaders import DataSplitter

# --- Project‑specific modules
# NOTE: keep sys.path adjustments *before* relative imports
sys.path.append(
    "/archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/run_models/scVI"
)
sys.path.append(
    "/archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart"
)

from model_config import *  # noqa: F403  (imports model_params_dict, build_model_dict, etc.)
from scMEDAL.utils.utils import (
    create_folder,
    read_adata,
    get_OHE,
    min_max_scaling,
    plot_rep,
    calculate_merge_scores,
    get_split_paths,
    calculate_zscores,
    get_clustering_scores_optimized,
)
from scMEDAL.utils.callbacks import ComputeLatentsCallback
from scMEDAL.utils.model_train_utils import ModelManager, load_data  #, compute_scores

# Silence potential tensorflow warnings for cleaner logs
warnings.filterwarnings("ignore", category=FutureWarning)
print("tf_version", tf.__version__)

import time


/tmp/ipykernel_53056/3802417333.py:7: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/archive/bioi

data_base_path: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../data/HealthyHeart_data
outputs_path: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../outputs/HealthyHeart_outputs
Save model set to: True
Parent folder: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../data/HealthyHeart_data/log_transformed_3000hvggenes/splits
Run name: run_latent_classifier_fe_latent-scMEDAL-FE_latent_2025-05-21_21-15
tf_version 2.19.0


I0000 00:00:1747880129.343506   53056 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:82:00.0, compute capability: 6.0


In [15]:

from paths_config import results_path_dict, input_base_path

from scMEDAL.utils.model_train_utils import (
    ModelManager,
    run_model_pipeline_LatentClassifier_v2,
    calculate_metrics_with_ci
)
from model_config import (
    data_path, model_params_dict, base_paths_dict, run_name,
    LatentClassifier_config, load_latent_spaces_dict,
    saved_models_base, source_file
)

from scMEDAL.utils.compare_results_utils import (
    get_input_paths_df,
    get_latent_paths_df,
    create_latent_dict_from_df
)
# from scMEDAL.models.scMEDAL import MixedEffectsModel


In [16]:

# --------------------------------------------------------------------------------------
# 1. Get Input Paths and Latent Paths
# --------------------------------------------------------------------------------------
df_latent = get_latent_paths_df(results_path_dict, k_folds=5)
df_inputs = get_input_paths_df(input_base_path, k_folds=5, eval_test=True)

# Merge latent and input paths
df = pd.merge(df_latent, df_inputs, on=["Split", "Type"], how="left")
print("Reading paths,\ndf paths:\n", df.head(5))

# --------------------------------------------------------------------------------------
# 2. Define Variables Necessary to Load Data and Train Model
# --------------------------------------------------------------------------------------
batch_col = model_params_dict['batch_col']
bio_col = model_params_dict['bio_col']

# Load metadata
metadata_all = pd.read_csv(glob.glob(os.path.join(data_path, "*meta.csv"))[0])
metadata_all[bio_col] = metadata_all[bio_col].astype('category')

# Define batch column (original metadata does not include batch yet)
metadata_all[batch_col] = metadata_all["sampleID"].astype('category')

print("n batches:", len(np.unique(metadata_all[batch_col]).tolist()))

# Define categories for batch and bio columns
batch_col_categories = np.unique(metadata_all[batch_col]).tolist()
print("check ordered batches:", batch_col_categories)

bio_col_categories = np.unique(metadata_all[bio_col]).tolist()
print("bio categories:", bio_col_categories)

# --------------------------------------------------------------------------------------
# 3. Update the Config for the Model
# --------------------------------------------------------------------------------------
load_latent_spaces_dict['batch_col_categories'] = batch_col_categories
load_latent_spaces_dict['bio_col_categories'] = bio_col_categories

# Update model
LatentClassifier_config['Model'] = MixedEffectsModel

# Create latent path dictionary
latent_path_dict = create_latent_dict_from_df(df_latent)
load_latent_spaces_dict["latent_path_dict"] = latent_path_dict

Directory does not exist: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../outputs/HealthyHeart_outputs/latent_space/log_transformed_3000hvggenes/scMEDAL-RE/scMEDAL-RE_run_name/splits_1
Directory does not exist: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../outputs/HealthyHeart_outputs/latent_space/log_transformed_3000hvggenes/scMEDAL-RE/scMEDAL-RE_run_name/splits_2
Directory does not exist: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../outputs/HealthyHeart_outputs/latent_space/log_transformed_3000hvggenes/scMEDAL-RE/scMEDAL-RE_run_name/splits_3
Directory does not exist: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../outputs/HealthyHeart_outputs/latent_space/log_transformed_3000hvggenes/scMEDAL-RE/scMEDAL-RE_run_name/splits_4
Directory does not exist: /archive/b

KeyError: 'Split'

In [4]:
# --------------------------------------------------------------------------------------
# 4. Run the Classifier for All Folds Latent Space
# --------------------------------------------------------------------------------------
folds_list = [1] #list(range(1, 6))  # Folds 1 to 5
all_folds_metrics_df = pd.DataFrame()

for fold in folds_list:
    print("fold", fold)
    load_latent_spaces_dict["fold"] = fold

    # Initialize model manager
    model_manager = ModelManager(
        model_params_dict,
        base_paths_dict,
        run_name,
        save_model=LatentClassifier_config["save_model"],
        use_kfolds=True,
        kfold=load_latent_spaces_dict["fold"]
    )

    # Update LatentClassifier config
    load_latent_spaces_dict["model_params"] = model_manager.params
    pipeline_LatentClassifier_config = {**load_latent_spaces_dict, **LatentClassifier_config}

fold 1
Created folder: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../outputs/HealthyHeart_outputs/saved_models/log_transformed_3000hvggenes/MEC/run_latent_classifier_fe_latent-scMEDAL-FE_latent_2025-05-21_21-15/splits_1
Created folder: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../outputs/HealthyHeart_outputs/figures/log_transformed_3000hvggenes/MEC/run_latent_classifier_fe_latent-scMEDAL-FE_latent_2025-05-21_21-15/splits_1
Created folder: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../outputs/HealthyHeart_outputs/latent_space/log_transformed_3000hvggenes/MEC/run_latent_classifier_fe_latent-scMEDAL-FE_latent_2025-05-21_21-15/splits_1
Folder already exists: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../outputs/HealthyHeart_outputs/saved_models/log_transformed_

In [14]:
config = pipeline_LatentClassifier_config

# Load initial dataset paths and data
input_path_dict = get_split_paths(base_path=config['base_path'], fold=config['fold'])
adata_dict = load_data(
    input_path_dict,
    eval_test=config['model_params'].eval_test,
    scaling=config['model_params'].scaling,
    issparse=config['issparse'],
    load_dense=config['load_dense']
)

# Initialize dataset list and add 'test' dataset if evaluation on 'test' is enabled
dataset_list = ["train", "val"]
if config['model_params'].eval_test:
    dataset_list.append("test")

# Iterate through each model and dataset
for model in config['models_list']:
    for dataset in dataset_list:
        # Load the latent space for the current model and dataset
        print(config['latent_path_dict'])
        # print(config['latent_path_dict'][model]["splits_" + str(config['fold'])][dataset])
        # latent_space_path = config['latent_path_dict'][model]["splits_" + str(config['fold'])][dataset]


Reading data from: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../data/HealthyHeart_data/log_transformed_3000hvggenes/splits/split_1/train
X.shape before scaling (291680, 3000)
Reading data from: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../data/HealthyHeart_data/log_transformed_3000hvggenes/splits/split_1/val


/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


X.shape before scaling (97227, 3000)
Reading data from: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../data/HealthyHeart_data/log_transformed_3000hvggenes/splits/split_1/test


/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


X.shape before scaling (97227, 3000)


/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


None
None
None
None
None
None


In [5]:
pipeline_LatentClassifier_config

{'latent_path_dict': None,
 'model_params': <scMEDAL.utils.model_train_utils.ModelManager.Namespace at 0x2aacf4493e60>,
 'base_path': '/archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../data/HealthyHeart_data/log_transformed_3000hvggenes/splits',
 'fold': 1,
 'models_list': ['scMEDAL-RE', 'scVI'],
 'batch_col_categories': None,
 'bio_col_categories': None,
 'issparse': True,
 'load_dense': True,
 'batch_col': 'batch',
 'bio_col': 'celltype',
 'Model': None,
 'build_model_dict': {'n_latent_dims': 2,
  'layer_units': [8, 4],
  'n_pred': 13,
  'add_re_2_meclass': False,
  'name': 'mec'},
 'compile_dict': {'optimizer': <keras.src.optimizers.adam.Adam at 0x2aacf1cc8ec0>,
  'loss': <LossFunctionWrapper(<function categorical_crossentropy at 0x2aacf4538680>, kwargs={'from_logits': False, 'label_smoothing': 0.0, 'axis': -1})>,
  'loss_weights': 1,
  'metrics': [<CategoricalAccuracy name=accuracy>]},
 'save_model': True,
 'latent_keys_config':

In [6]:
config = pipeline_LatentClassifier_config
config['models_list']

['scMEDAL-RE', 'scVI']

In [ ]:
def get_x_y_z(adata, batch_col, bio_col, batch_col_categories, bio_col_categories, use_rep="X"):
    """
    Extracts and returns features, labels, and batch information from an AnnData object.

    Parameters:
    - adata: AnnData object containing the dataset.
    - batch_col (str): The name of the batch column.
    - bio_col (str): The name of the biological column.
    - batch_col_categories (list): Categories for the batch column to be used in one-hot encoding.
    - bio_col_categories (list): Categories for the biological column to be used in one-hot encoding.
    - use_rep (str, optional): The representation of data to be used. 'X' for default representation or key for .obsm.

    Returns:
    - x_y_z_dict (dict): Dictionary containing:
        - 'x': Features from the AnnData object (either .X or specified .obsm).
        - 'y': One-hot encoded labels.
        - 'z': One-hot encoded batch information.
    """

    # Choose the data representation based on 'use_rep'
    if use_rep != "X" and use_rep not in adata.obsm:
        raise KeyError(f"The key '{use_rep}' is not found in adata.obsm. Available keys: {list(adata.obsm.keys())}")
    x = adata.X if use_rep == "X" else adata.obsm[use_rep]

    # Retrieve one-hot encoded batch information
    z_ohe = get_OHE(adata, categories=batch_col_categories, col=batch_col)

    # Retrieve one-hot encoded labels
    y_ohe = get_OHE(adata, categories=bio_col_categories, col=bio_col)

    # Create a dictionary to store 'x', 'y', and 'z' components
    x_y_z_dict = {
        "x": x,
        "y": y_ohe,
        "z": z_ohe
    }

    return x_y_z_dict

def load_latent_spaces(base_path, fold, models_list, latent_path_dict, model_params, batch_col, bio_col, batch_col_categories, bio_col_categories,issparse=False, load_dense=False):
    """
    Loads and stores latent spaces for specified models and datasets, and retrieves 'y' and 'z' components. This is useful for retriving the save latent space after applying: AE_DA,AEC_DA,AE_RE,AE_conv
    Parameters:
    - base_path (str): Base path to the data.
    - fold (int): The specific fold of the data.
    - models_list (list): List of model names.
    - latent_path_dict (dict): Dictionary mapping models and datasets to their latent space paths. You can obtain it using the function get_latent_spaces_paths.
    - model_params (Namespace object): The parameters of the model, used for scaling loss values. Created using ModelManager class.
    - batch_col (str): The batch column name used for retrieving 'z' component.
    - bio_col (str): The biological column name used for retrieving 'y' component.
    - batch_col_categories (list): Categories for the batch column to be used in one-hot encoding.
    - bio_col_categories (list): Categories for the biological column to be used in one-hot encoding.
    - issparse(bool): True if X is in sparse array, False if its dense
    - load_dense (bool): If True, forces conversion of sparse arrays to dense format.



    Returns:
    - adata_dict (dict): Dictionary of AnnData objects updated with latent spaces and 'y', 'z' components.
    """
    
    # Load initial dataset paths and data
    input_path_dict = get_split_paths(base_path=base_path, fold=fold)
    adata_dict = load_data(input_path_dict, eval_test = model_params.eval_test, scaling = model_params.scaling,issparse=issparse, load_dense=load_dense)

    # Initialize dataset list and add 'test' dataset if evaluation on 'test' is enabled
    dataset_list = ["train", "val"]
    if model_params.eval_test:
        dataset_list.append("test")
        
    # Iterate through each model and dataset
    for model in models_list:
        for dataset in dataset_list:
            # Load the latent space for the current model and dataset
            latent_space_path = latent_path_dict[model]["splits_" + str(fold)][dataset]
            if isinstance(latent_space_path, (np.ndarray,list)):
                latent_space_path = latent_space_path[0]
            latent_space = np.load(latent_space_path)

            latent_key = f"{model}_latent_{dataset}"
            # save latent space in .obsm as latent_key
            adata_dict[dataset].obsm[latent_key] = latent_space
  

    # Iterate through datasets to retrieve 'y' and 'z' components
    for dataset in dataset_list:
        x_y_z_dict = get_x_y_z(adata_dict[dataset], batch_col, bio_col, 
                               batch_col_categories, bio_col_categories, use_rep="X")
        adata_dict[dataset+'_y'] = x_y_z_dict['y']
        adata_dict[dataset+'_z'] = x_y_z_dict['z']

    return adata_dict

In [7]:
from scMEDAL.utils.model_train_utils import load_latent_spaces
config = pipeline_LatentClassifier_config

adata_dict = load_latent_spaces(
    base_path=config['base_path'],
    fold=config['fold'],
    models_list=config['models_list'],
    latent_path_dict=config['latent_path_dict'],
    model_params=config['model_params'],
    batch_col=config['batch_col'],
    bio_col=config['bio_col'],
    batch_col_categories=config['batch_col_categories'],
    bio_col_categories=config['bio_col_categories'],
    issparse=config['issparse'],
    load_dense=config['load_dense']
)


Reading data from: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../data/HealthyHeart_data/log_transformed_3000hvggenes/splits/split_1/train
X.shape before scaling (291680, 3000)
Reading data from: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../data/HealthyHeart_data/log_transformed_3000hvggenes/splits/split_1/val


/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


X.shape before scaling (97227, 3000)
Reading data from: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../data/HealthyHeart_data/log_transformed_3000hvggenes/splits/split_1/test


/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


X.shape before scaling (97227, 3000)


/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


TypeError: 'NoneType' object is not subscriptable

In [8]:
latent_path_dict

NameError: name 'latent_path_dict' is not defined

In [11]:
config['models_list']

['scMEDAL-RE', 'scMEDAL-FE', 'scMEDAL-FEC', 'AEC', 'AE']

In [ ]:
latent_path_dict[model]["splits_" + str(fold)][dataset]

In [ ]:

# 1. Load data latent paths and adata_dict
adata_dict = load_latent_spaces(base_path, fold, models_list, latent_path_dict, model_params, batch_col, bio_col, batch_col_categories, bio_col_categories,issparse, load_dense)

In [7]:
vars(load_latent_spaces_dict["model_params"])

{'optimizer': <keras.src.optimizers.adam.Adam at 0x2aacf4a08d70>,
 'loss': <LossFunctionWrapper(<function categorical_crossentropy at 0x2aacf49d5da0>, kwargs={'from_logits': False, 'label_smoothing': 0.0, 'axis': -1})>,
 'loss_weights': 1,
 'metrics': [<CategoricalAccuracy name=accuracy>],
 'n_latent_dims': 2,
 'layer_units': [8, 4],
 'n_pred': 13,
 'add_re_2_meclass': False,
 'name': 'mec',
 'eval_test': True,
 'scaling': None,
 'batch_size': 512,
 'epochs': 2,
 'monitor_metric': 'val_loss',
 'patience': 30,
 'stop_criteria': 'early_stopping',
 'compute_latents_callback': False,
 'encoder_latent_name': 'MEC_latent',
 'get_pca': False,
 'get_baseline': False,
 'batch_col': 'batch',
 'bio_col': 'celltype',
 'model_path_main': '/archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../outputs/HealthyHeart_outputs/saved_models/log_transformed_3000hvggenes/MEC/run_latent_classifier_fe_latent-scMEDAL-FE_latent_2025-05-21_17-24',
 'plots_path_mai

In [ ]:
cv_results = run_scvi_across_folds(
    input_base_path        = input_base_path,
    out_base_paths_dict    = base_paths_dict,
    folds_list             = folds_list,
    run_name               = run_name,
    model_params_dict      = model_params_dict,
    build_model_dict       = build_model_dict,
    compile_dict           = compile_dict,
    save_model             = save_model,
    batch_col              = batch_col,
    bio_col                = bio_col,
    batch_col_categories   = seen_donor_ids,
    bio_col_categories     = celltype_ids,
    model_type             = "scvi",
    issparse               = True,
    load_dense             = True,
    shape_color_dict       = shape_color_dict,
    return_scores_temp     = True,
    sample_size = 10000,
    n_batch2plot = 20,
    seed = 5,
    plot_params = plot_params
    )

In [3]:
base_paths_dict

{'models': '/archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../outputs/HealthyHeart_outputs/saved_models/log_transformed_3000hvggenes/MEC',
 'figures': '/archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../outputs/HealthyHeart_outputs/figures/log_transformed_3000hvggenes/MEC',
 'latent': '/archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../outputs/HealthyHeart_outputs/latent_space/log_transformed_3000hvggenes/MEC'}

In [2]:
# split train RE 2  epochs (from the paper)
path_latent_RE_500epochs ="/archive/bioinformatics/DLLab/AixaAndrade/results/mixedeffectsdl/results/ARMED_genomics/heart_data/outputs/latent_space/Healthy_human_heart_data/log_transformed_3000hvggenes/AE_RE/run_crossval_loss_recon_weight-110.0_loss_latent_cluster_weight-0.1_n_latent_dims-2_layer_units-512-132_kl_weight-0.0_scaling-min_max_batch_size-512_epochs-500_patience-30_sample_size-10000_2024-07-24_13-48/splits_1/RE_AE_latent_2_"
# split train RE 2  epochs (it will not work that well)
path_latent_RE_2epochs = "/archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/outputs/HealthyHeart_outputs/latent_space/log_transformed_3000hvggenes/scMEDAL-RE/run_crossval_loss_recon_weight-110.0_loss_latent_cluster_weight-0.1_n_latent_dims-2_layer_units-512-132_kl_weight-0.0_scaling-min_max_batch_size-512_epochs-2_patience-30_sample_size-10000_2024-12-12_17-39/splits_1/scMEDAL-RE_latent_2_"
# split train scVI 2 epochs
path_latent_scVI = "/archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/outputs/HealthyHeart_outputs/latent_space/log_transformed_3000hvggenes/scVI/run_crossval_n_latent_dims-2_n_layers-2_n_hidden-128_gene_likelihood-zinb_dispersion-gene_scaling-min_max_batch_size-512_epochs-2_patience-30_compute_latents_callback-False_sample_size-10000_model_type-ae_n_components-2_2025-05-07_22-19/splits_1/z_"

In [3]:
latent_train_RE_500epochs = np.load(f"{path_latent_RE_500epochs}train.npy")
latent_train_RE_2epochs = np.load(f"{path_latent_RE_2epochs}train.npy")
latent_train_scVI  = np.load(f"{path_latent_scVI}train.npy")

In [4]:
latent_train_RE_500epochs.shape

(291680, 2)

In [5]:
latent_train_RE_2epochs.shape

(291680, 2)

In [6]:
latent_train_scVI.shape

(291680, 2)

In [ ]:
#Now train on scVI alone and try scVI +RE

In [4]:

def run_model_pipeline_LatentClassifier_v2_PCA(Model, latent_path_dict, build_model_dict, compile_dict, model_params, save_model, 
                                           batch_col, bio_col, base_path, fold, models_list, latent_keys_config,
                                           batch_col_categories=None, bio_col_categories=None, return_metrics=True, 
                                           return_adata_dict=False, return_trained_model=False, model_type="mec",
                                           issparse=False, load_dense=False,seed=None):
    """
    Runs the complete model pipeline, including data loading, model training, evaluation, and metric collection.

    Parameters:
    - Model: The model class to be instantiated and trained.
    - latent_path_dict: Dictionary containing paths to latent space data for each model.
    - build_model_dict: Dictionary of parameters for building the model.
    - compile_dict: Dictionary of parameters for compiling the model.
    - model_params: Object containing additional model parameters and configurations.
    - save_model: Boolean flag indicating whether to save the trained model to disk.
    - batch_col: Name of the column representing batch information in the data.
    - bio_col: Name of the column representing biological information in the data.
    - base_path: Base directory path for datasets and model output.
    - fold: The specific fold identifier for cross-validation or data splitting.
    - models_list: List of models to be used in the pipeline.
    - latent_keys_config: Configuration dictionary for the latent keys used in model input.
    - batch_col_categories: List or array of categories for the batch column (optional).
    - bio_col_categories: List or array of categories for the biological column (optional).
    - return_metrics: Boolean flag indicating whether to return performance metrics (default: True).
    - return_adata_dict: Boolean flag indicating whether to return the AnnData dictionary (default: False).
    - return_trained_model: Boolean flag indicating whether to return the trained model (default: False).
    - model_type: String specifying the type of model being used (default: "mec").
    - issparse(bool): True if X is in sparse array, False if its dense
    - load_dense (bool): If True, forces conversion of sparse arrays to dense format.
    - seed (int) : seed set for repreducible results of dummy classifier with strategy: stratified. Default: None

    Returns:
    - results: Dictionary containing the results based on the specified flags. Possible keys include:
        - "dffn_model": The trained deep feedforward network model (if return_trained_model is True).
        - "metrics": DataFrame of performance metrics for the trained model and SVM.
        - "adata": The AnnData dictionary containing processed data (if return_adata_dict is True).
    """

    # 1. Load data latent paths and adata_dict
    adata_dict = load_latent_spaces(base_path, fold, models_list, latent_path_dict, model_params, batch_col, bio_col, batch_col_categories, bio_col_categories,issparse, load_dense)
    # Calculate PCA
    adata_dict = get_pca_andplot(adata_dict, plot_params=None, eval_test=model_params.eval_test,n_components=model_params.n_components,shape_color_dict = None)

    print("Batches available: ", np.unique(adata_dict["train"].obs[batch_col]))

    # 2. Prepare data for training
    inputs = prepare_latent_space_inputs(adata_dict, latent_keys_config, eval_test=model_params.eval_test)

    # 3. Build and train model, plott loss and evaluate dffn model
    dffn_results = build_train_evaluate_model(Model,build_model_dict, compile_dict, inputs, adata_dict, model_params, save_model, model_type)

    adata_dict = dffn_results["adata_dict"]
    dffn_metrics = dffn_results["metrics"]
    print("Training svc classifier")

    svm_results = svm_accuracy_and_predictions(inputs, adata_dict, model_params,eval_test=model_params.eval_test)
    adata_dict = svm_results["adata_dict"]
    svm_metrics = svm_results["metrics"]
    #metrics_df = pd.merge(dffn_metrics,svm_metrics)


    # Evaluate using RandomForest
    print("Training random forest classifier")

    rf_results = random_forest_accuracy_and_predictions(inputs, adata_dict, model_params, eval_test=model_params.eval_test)
    adata_dict = rf_results["adata_dict"]
    rf_metrics = rf_results["metrics"]

    # Calculate chance accuracy
    chance_results = dummy_classifier_chance_accuracy(inputs, adata_dict, model_params, eval_test=model_params.eval_test,seed=seed)
    adata_dict = chance_results["adata_dict"]
    chance_metrics = chance_results["metrics"]

    # Merge the metrics from DFFN, SVM, and RandomForest
    metrics_df = pd.merge(dffn_metrics, svm_metrics, on="Dataset")
    metrics_df = pd.merge(metrics_df, rf_metrics, on="Dataset")
    metrics_df = pd.merge(metrics_df, chance_metrics, on="Dataset")

    metrics_df.to_csv(os.path.join(model_params.latent_path, "metrics.csv"))

    


    # 7. Collect results based on flags
    results = {}
    if return_trained_model:
        results["dffn_model"] = dffn_results["model"]
    if return_metrics:
        results["metrics"] = metrics_df
    if return_adata_dict:
        results["adata"] = adata_dict

    return results

In [ ]:
# --------------------------------------------------------------------------------------
# 4. Run the Classifier for All Folds Latent Space
# --------------------------------------------------------------------------------------
folds_list = list(range(1, 6))  # Folds 1 to 5
all_folds_metrics_df = pd.DataFrame()

for fold in folds_list:
    print("fold", fold)
    load_latent_spaces_dict["fold"] = fold

    # Initialize model manager
    model_manager = ModelManager(
        model_params_dict,
        base_paths_dict,
        run_name,
        save_model=LatentClassifier_config["save_model"],
        use_kfolds=True,
        kfold=load_latent_spaces_dict["fold"]
    )

    # Update LatentClassifier config
    load_latent_spaces_dict["model_params"] = model_manager.params
    pipeline_LatentClassifier_config = {**load_latent_spaces_dict, **LatentClassifier_config}